# 03 — Time axis & relationships to the target (team 07)

**Owner:** Ranbin · **Branch:** `team-07-ranbin`

**Questions to answer**
1. **Is there a real time axis?** (Unlike the Kaggle data, this one should have one.) What's the
   date range and granularity? Any gaps (missing months, COVID-era drops)?
2. **Trend & seasonality** — plot total `TARGET` over time; then a few individual routes.
   Is there a yearly pattern? This decides whether we need lag/season features or a time-series model.
3. **Relationships** — scatter/correlation of numeric features vs the target. Linear or non-linear?
   (Non-linear points toward trees; linear toward Ridge/Lasso.)
4. What does this mean for the **train/test split**? (Time data → split by date, not randomly.)

**Write the "why", not just the plot.** Every plot or number needs a sentence after it:
*"…therefore we should ___ when we model."* Those sentences go in the table at the bottom.

> **NDA reminder.** This is real American Airlines data. Never commit it, never paste rows or
> screenshots of it anywhere outside the team, and **clear outputs before every commit**
> (Kernel → Restart & Clear Output) so no data ends up in the notebook file.

In [ ]:
%run ./00_load_and_preprocess.ipynb
# gives you: pax, airports, TARGET, REPO_ROOT, RESULTS_DIR

## 1. Time axis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 00 gives separate DEP_YEAR / DEP_MONTH columns; combine them into one month-start date
pax['DATE'] = pd.to_datetime(dict(year=pax['DEP_YEAR'], month=pax['DEP_MONTH'], day=1))

# 1. Date range check
min_date = pax['DATE'].min()
max_date = pax['DATE'].max()
print(f"Date range: {min_date.strftime('%Y-%m')} to {max_date.strftime('%Y-%m')}")

# 2. Count rows per monthly period and check for missing months
period_counts = pax.groupby('DATE').size().asfreq('MS', fill_value=0)

missing_months = period_counts[period_counts == 0]
print(f"Total months in span: {len(period_counts)}")
print(f"Missing months count: {len(missing_months)}")
if len(missing_months) > 0:
    print("Missing periods:", missing_months.index.strftime('%Y-%m').tolist())

# 3. Clean line plot with proper date axis
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(period_counts.index, period_counts.values, marker='o', markersize=3, color='steelblue', linewidth=1.5)

# Format the X-axis for readable dates
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))

ax.set_title(f"Observations (Rows) per Month ({min_date:%Y} - {max_date:%Y})")
ax.set_xlabel("Year")
ax.set_ylabel("Row Count")
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

What this shows:
* Temporal Boundary & Completeness: The dataset spans 91 consecutive months from January 2019 through July 2026 with 0 missing months.
* Baseline Route Stability: Outside of the pandemic shock, monthly observation counts are flat and stable, averaging 17100 rows per month across 2023–2025. This indicates a fixed set of monitored market routes rather than growing route-network expansion.
* COVID Structural Shock: Route reporting collapsed in early 2020, reaching a trough of 13000 rows in May 2020, followed by a recovery curve until full stabilization in mid-2022.

What this means for the model:
* Validation Strategy: Because we have a clear, continuous temporal structure (2019–2026), standard random K-Fold cross-validation will cause severe temporal data leakage. We must evaluate models using out-of-time evaluation or TimeSeriesSplit.
* Structural Handling (2020–2021 Data): The demand collapse and route cancellations in 2020–2021 represent an external shock, not typical passenger demand behavior. If included unadjusted in training, tree models or regressions will learn distorted price/route elasticities. The team should either:
* * Exclude 2020–2021 observations from model training or
* * Introduce a binary pandemic-regime flag to allow tree-based splits to isolate this regime.

## 2. Trend & seasonality

In [ ]:
import seaborn as sns

# 1. System-wide total passenger volume over time
monthly_target = pax.groupby('DATE')[TARGET].sum()

plt.figure(figsize=(10, 4))
monthly_target.plot(marker='o', color='navy')
plt.title(f"Total System-Wide {TARGET} Over Time")
plt.xlabel("Date")
plt.ylabel(f"Total {TARGET}")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2. Top 5 example routes
pax['ROUTE'] = pax['ORG_AIRPRT'] + '-' + pax['DES_AIRPRT']
top_5_routes = pax.groupby('ROUTE')[TARGET].sum().nlargest(5).index.tolist()
print("Top 5 routes analyzed:", top_5_routes)

sample_routes = pax[pax['ROUTE'].isin(top_5_routes)]

plt.figure(figsize=(11, 5))
sns.lineplot(data=sample_routes, x='DATE', y=TARGET, hue='ROUTE', marker='o')
plt.title(f"{TARGET} Trend Across Top 5 Routes")
plt.xlabel("Date")
plt.ylabel(TARGET)
plt.legend(title="Route", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

What this shows:

* Pronounced Annual Seasonality: Looking at the stabilized periods (2019 and post-2022), there is an cyclical  pattern in passenger demand. Demand surges predictably every summer (peaking around July) and troughs every winter/early spring (reaching lowest levels around February).
* Co-movement: The top 5 routes (JFK-LAX, LAX-JFK, LGA-ORD, ORD-LGA, SFO-LAX) move in near lockstep with the macro system-wide trend.
* Long-Term Demand Ceiling & Recovery: Following the steep 2020 drop, passenger volume recovered steadily through 2021–2023. By 2024–2026, total volume fully rebounded and plateaued near 2019 peak capacity, showing that the system has transitioned back to a steady-state cyclical regime.

What it means for the model:

* Cyclical Feature: Because DEP_MONTH has a non-linear cyclical relationship with demand (December/January are adjacent, but numerically 12 and 1), we should provide the model with cyclical sine/cosine encodings or categorical month indicators.
*  Lag / Rolling Demand Features: Because routes show high auto-correlation and seasonal persistence, historical lag features will likely be top-tier predictors if our inference setup allows them. 
* Tree Extrapolation Limits: Tree-based models (LightGBM/XGBoost) cannot extrapolate positive growth trends beyond the training set range. However, because post-2023 volume has plateaued into a cyclical steady state rather than exponential growth, tree models will handle this stationary seasonality well without requiring trend-differencing.  

## 3. Relationships

In [ ]:
# Select relevant numeric predictors
numeric_cols = ['DEP_MONTH', 'DEP_YEAR', 'OD_REV_USD']

# Spearman rank correlation
corr_series = pax[numeric_cols + [TARGET]].corr(method='spearman')[TARGET].drop(TARGET)
print("Spearman Rank Correlation with TARGET:\n")
print(corr_series)

# Scatter plot: TARGET vs OD_REV_USD (log scale)
plt.figure(figsize=(6, 5))
sample_pax = pax.sample(n=min(5000, len(pax)), random_state=42)

plt.scatter(sample_pax['OD_REV_USD'], sample_pax[TARGET], alpha=0.3, s=15, color='teal')
plt.xscale('log')
plt.yscale('log')
plt.xlabel("OD_REV_USD (log scale)")
plt.ylabel(f"{TARGET} (log scale)")
plt.title(f"{TARGET} vs OD_REV_USD")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

What this shows:
* Extreme Target Correlation (r_s = 0.946): OD_REV_USD has a near-perfect Spearman rank correlation of +0.946 with OD_PAX. The log-log scatter plot confirms a tight, linear relationship across multiple orders of magnitude. 
* OD_REV_USD is an after event realized outcome
* Weak Raw Linear Date Effects: DEP_MONTH (r_s = 0.026) and DEP_YEAR (r_s = 0.141) show virtually zero monotonic linear correlation with OD_PAX. Because seasonality is cyclical (non-monotonic) and the trend had a severe structural drop (2020) rather than a linear trajectory, treating DEP_MONTH or DEP_YEAR as standard numeric linear features fails to capture their actual relationship.

What it means for the model:
* Do not use OD_REV_USD as a feature but could use OD_REV_USD/OD_PAX (yield) as an alternative and lag it behind by 3-12 months as a pricing feature. 
* Log Transformations: The log-log linearity between revenue and demand confirms that passenger volume operates on a multiplicative/log scale across routes. When modeling OD_PAX, we should evaluate training on log(1 + OD_PAX) or using Poisson/Tweedie objectives in LightGBM/XGBoost to handle the wide dynamic range.

## Findings → modelling consequence (Ranbin)

This table is what the team's feature set (05) is built from. Every row needs a **consequence**,
not just an observation — "*therefore we should ___ when we model.*"

| # | Observation (with the number) | Modelling consequence |
|---|---|---|
| 1 | 91 consecutive months (2019-01 through 2026-07) with 0 missing months. Stable route coverage (~17k rows/mo) except a steep COVID-19 collapse in 2020. | Use time-based validation (TimeSeriesSplit or 2025–2026 holdout) instead of random K-Fold. Filter out or isolate 2020–2021 with a regime flag so the model does not learn distorted pandemic relationships. |
| 2 | Strong annual cyclicality with summer volume peaks and winter troughs. Top routes mirror overall macro system demand. Post-2023 demand plateaued near 2019 peak capacity. | Engineer cyclical encodings (sin/cos) or categorical dummy indicators for DEP_MONTH. Create 12-month seasonal lag features per route to anchor baseline demand. |
| 3 | OD_REV_USD has an extreme rank correlation of +0.946 with OD_PAX and log-log linearity. DEP_MONTH has near-zero linear correlation (r_s = 0.026) due to cyclicality. | Drop OD_REV_USD from the feature set to prevent target leakage (revenue is realized ex-post from passenger sales). Consider predicting log(OD_PAX) or using a Tweedie loss function to account for heavy-tailed route volumes. |